In [ ]:
# [fugu-core]
import os
from dataclasses import dataclass
from typing import Any
from pydantic import BaseModel, Field

PROVIDER, MODEL = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai"), os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")
PROVIDER_ENV = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "gemini": "GEMINI_API_KEY", "xai": "XAI_API_KEY", "deepseek": "DEEPSEEK_API_KEY"}
TOKEN_BUDGET, MAX_COST, COST_PER_MTOK, MAX_CALLS, MAX_TOOL_ROUNDS = 20_000, 0.30, 4.0, 12, 5

ORCHESTRATOR_PROMPT = """You are Fugu, an orchestration model. Return a typed plan, not an answer. Use mode='direct' for simple single-pass asks. Use mode='delegate' for hard, multi-step, or multi-domain asks; a team of experts (reasoning, coding, long_context, fast) will each tackle the request and their outputs will be synthesized."""
SYNTHESIZER_PROMPT = """Combine the original request and labeled expert outputs into one complete answer. Reconcile conflicts, remove redundancy, and do not mention the orchestration machinery."""
VERIFIER_PROMPT = """Check the draft against the original request, fix errors or gaps, and return only the final answer."""
EXPERT_PROMPT = """You are a {specialty} expert inside Fugu. Solve the request completely. Use search/read_article for factual grounding when useful."""

class OrchestrationPlan(BaseModel):
    mode: str = Field(description="direct or delegate")
    reasoning: str = Field(description="Why this route was chosen.")

@dataclass
class PoolModel:
    key: str; provider: str; model: str; agent: Any

class Fugu:
    def __init__(self, orchestrator, pool, synthesizer, verifier=None, *, ultra=False, exclude=()):
        self.orchestrator, self.pool, self.synthesizer, self.verifier, self.ultra = orchestrator, [m for m in pool if m.key not in exclude], synthesizer, verifier, ultra
    def run(self, prompt):
        return self.orchestrate(prompt)
    def orchestrate(self, prompt):
        try:
            payload = self.orchestrator.run(prompt).metadata.get("structured")
            plan = payload if isinstance(payload, OrchestrationPlan) else OrchestrationPlan.model_validate(payload)
        except Exception:
            plan = OrchestrationPlan(mode="direct", reasoning="fallback: no structured plan")
        available = [m for m in self.pool if os.getenv(PROVIDER_ENV.get(m.provider, ""))]
        if not available:
            raise RuntimeError("Fugu: no available models in pool")
        if plan.mode != "delegate":
            return available[0].agent.run(prompt).content
        outputs = [f"[{m.key}] {m.agent.run(prompt).content}" for m in available]
        answer = self.synthesizer.run(f"Original request:\n{prompt}\n\nExpert outputs:\n" + "\n\n".join(outputs)).content
        if self.ultra and self.verifier:
            return self.verifier.run(f"Request:\n{prompt}\n\nDraft answer:\n{answer}").content
        return answer

try:
    import re, requests
    from vidbyte import BaseAgent, tool
    from vidbyte.middleware import TokenBudgetMiddleware, CostBudgetMiddleware, RuntimeLimitMiddleware, ModelRetryMiddleware
    @tool
    def search(query: str) -> list[dict]:
        r = requests.get("https://en.wikipedia.org/w/api.php", headers={"User-Agent": "vidbyte-cookbook-sakana-fugu/0.1"}, timeout=30, params={"action": "query", "list": "search", "srsearch": query, "srlimit": 5, "format": "json"}); r.raise_for_status()
        return [{"title": h["title"], "snippet": re.sub(r"<[^>]+>", "", h["snippet"])} for h in r.json()["query"]["search"]]
    @tool
    def read_article(title: str) -> str:
        r = requests.get("https://en.wikipedia.org/w/api.php", headers={"User-Agent": "vidbyte-cookbook-sakana-fugu/0.1"}, timeout=30, params={"action": "query", "prop": "extracts", "explaintext": 1, "titles": title, "format": "json", "redirects": 1}); r.raise_for_status()
        return (next(iter(r.json()["query"]["pages"].values())).get("extract") or f"No article found for {title!r}.")[:6000]
    def guards(): return [TokenBudgetMiddleware(max_tokens=TOKEN_BUDGET), CostBudgetMiddleware(max_spend_usd=MAX_COST, cost_per_million_tokens=COST_PER_MTOK), RuntimeLimitMiddleware(max_model_calls=MAX_CALLS, max_elapsed_seconds=120), ModelRetryMiddleware(max_attempts=2)]
    def agent(name, prompt, **kw): return BaseAgent(name=name, system_prompt=prompt, provider=PROVIDER, model_name=MODEL, middleware=guards(), **kw)
    orchestrator = agent("fugu-orchestrator", ORCHESTRATOR_PROMPT, output_schema=OrchestrationPlan)
    synthesizer, verifier = agent("fugu-synthesizer", SYNTHESIZER_PROMPT), agent("fugu-verifier", VERIFIER_PROMPT)
    def expert(key, provider, model, specialty): return PoolModel(key, provider, model, BaseAgent(name=f"expert-{key}", system_prompt=EXPERT_PROMPT.format(specialty=specialty), provider=provider, model_name=model, tools=[search, read_article], middleware=guards(), max_tool_rounds=MAX_TOOL_ROUNDS))
    pool = [expert("reasoning", "anthropic", "claude-sonnet-4-6", "deep reasoning"), expert("coding", "openai", MODEL, "software engineering"), expert("long_context", "gemini", "gemini-2.5-pro", "long-context synthesis"), expert("fast", "deepseek", "deepseek-chat", "fast drafting")]
    fugu = Fugu(orchestrator, pool, synthesizer, verifier=verifier)
    print(fugu.run("Compare why Concorde retired with what a modern supersonic revival must solve."))
except ImportError:
    pass
